# UpLift — Notebook 06: Agency-Facing Summary Report
**Project:** UpLift — Predictive Maintenance for Transit Accessibility Equipment  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/uplift-transit](https://github.com/meyeringn/uplift-transit)

---

## What This Notebook Does

Notebooks 01–05 are for data scientists and technical reviewers. This notebook is for everyone else.

It produces a plain-language summary of UpLift's findings — written for transit agency leadership, maintenance supervisors, ADA compliance officers, and board members who need to understand what the model does and what it recommends, without needing to understand how it works.

**Outputs:**
- Executive summary table (key metrics in plain language)
- Top 20 highest-risk units formatted for a maintenance briefing
- Priority tier breakdown by agency and equipment type
- `uplift_agency_report.csv` — the operational deliverable a transit agency would actually use
- `uplift_executive_summary.md` — a Markdown summary ready to paste into any report or proposal

In [ ]:
# Install dependencies (run once in Google Colab)
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import fbeta_score, roc_auc_score, recall_score, precision_score

sns.set_theme(style='whitegrid')
THRESHOLD = 0.30
REPORT_DATE = datetime.today().strftime('%B %d, %Y')
print(f'Report date: {REPORT_DATE}')

## 1. Load Everything

In [ ]:
model = joblib.load('uplift_xgboost_model.pkl')
X_holdout = np.load('X_holdout.npy')
y_holdout = np.load('y_holdout.npy')
feature_cols = joblib.load('uplift_feature_cols.pkl')
encoders = joblib.load('uplift_encoders.pkl')
df_orig = pd.read_csv('uplift_equipment_data.csv')

# Score full dataset
df_model_input = df_orig.drop(columns=['equipment_id', 'outage_within_30_days'])
for col in ['equipment_type', 'agency', 'manufacturer']:
    df_model_input[col] = encoders[col].transform(df_model_input[col])
full_probs = model.predict_proba(df_model_input[feature_cols].values)[:, 1]

# Build agency report dataframe
report_df = df_orig[['equipment_id','equipment_type','agency','manufacturer',
                      'equipment_age_years','days_since_last_maintenance',
                      'unplanned_outages_12mo','inspection_score','is_transfer_hub']].copy()
report_df['risk_score'] = full_probs.round(3)
report_df['risk_flag'] = (full_probs >= THRESHOLD).astype(int)
report_df['priority_tier'] = pd.cut(
    full_probs,
    bins=[0, 0.30, 0.60, 0.80, 1.0],
    labels=['Monitor', 'Elevated', 'High', 'Critical']
)
report_df['recommended_action'] = report_df['priority_tier'].map({
    'Monitor':  'Schedule inspection at next routine cycle',
    'Elevated': 'Schedule inspection within 30 days',
    'High':     'Schedule proactive service call within 2 weeks',
    'Critical': 'Dispatch service team within 72 hours'
})
report_df = report_df.sort_values('risk_score', ascending=False).reset_index(drop=True)
report_df.to_csv('uplift_agency_report.csv', index=False)
print(f'Saved: uplift_agency_report.csv ({len(report_df)} units)')

## 2. Executive Summary — Key Numbers

In [ ]:
y_prob_holdout = model.predict_proba(X_holdout)[:, 1]
y_pred_holdout = (y_prob_holdout >= THRESHOLD).astype(int)

recall = recall_score(y_holdout, y_pred_holdout)
precision = precision_score(y_holdout, y_pred_holdout)
f2 = fbeta_score(y_holdout, y_pred_holdout, beta=2)
auc = roc_auc_score(y_holdout, y_prob_holdout)

flagged = report_df['risk_flag'].sum()
critical = (report_df['priority_tier'] == 'Critical').sum()
high = (report_df['priority_tier'] == 'High').sum()
elevated = (report_df['priority_tier'] == 'Elevated').sum()

summary = {
    'Total equipment units scored': len(report_df),
    'Units flagged for proactive service': int(flagged),
    '— Critical (dispatch within 72 hrs)': int(critical),
    '— High (service within 2 weeks)': int(high),
    '— Elevated (inspect within 30 days)': int(elevated),
    'Failures caught before they happen (Recall)': f'{recall:.1%}',
    'Accuracy when flagging a unit (Precision)': f'{precision:.1%}',
    'Primary metric — F2-Score': f'{f2:.3f}',
    'Overall model performance — AUC-ROC': f'{auc:.3f}',
    'Decision threshold': THRESHOLD,
    'Report generated': REPORT_DATE
}

print('=== UpLift Executive Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

## 3. Top 20 Highest-Risk Units — Maintenance Briefing Format

In [ ]:
top20 = report_df.head(20)[[
    'equipment_id', 'equipment_type', 'agency', 'risk_score',
    'priority_tier', 'recommended_action',
    'equipment_age_years', 'unplanned_outages_12mo', 'days_since_last_maintenance'
]].copy()
top20['equipment_age_years'] = top20['equipment_age_years'].round(1)
top20['days_since_last_maintenance'] = top20['days_since_last_maintenance'].round(0).astype(int)
top20.index = range(1, 21)
top20.index.name = 'Rank'

print('TOP 20 HIGHEST-RISK UNITS')
print('=' * 80)
print(top20.to_string())

## 4. Priority Tier Breakdown by Agency

In [ ]:
tier_by_agency = report_df.groupby(['agency', 'priority_tier']).size().unstack(fill_value=0)
tier_order = ['Monitor', 'Elevated', 'High', 'Critical']
tier_by_agency = tier_by_agency.reindex(columns=tier_order, fill_value=0)

colors = ['#81C784', '#FFB74D', '#EF5350', '#B71C1C']
fig, ax = plt.subplots(figsize=(12, 6))
tier_by_agency.plot(kind='bar', ax=ax, color=colors, alpha=0.9,
                    edgecolor='white', width=0.75)
ax.set_title('UpLift — Equipment Risk Tiers by Agency\n'
             f'Report Date: {REPORT_DATE} | Threshold: {THRESHOLD}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Agency', fontsize=11)
ax.set_ylabel('Number of Equipment Units', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Priority Tier', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('uplift_agency_tiers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_agency_tiers.png')

## 5. Priority Breakdown by Equipment Type

In [ ]:
tier_by_type = report_df.groupby(['equipment_type', 'priority_tier']).size().unstack(fill_value=0)
tier_by_type = tier_by_type.reindex(columns=tier_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
tier_by_type.plot(kind='bar', ax=ax, color=colors, alpha=0.9,
                  edgecolor='white', width=0.65)
ax.set_title('UpLift — Equipment Risk Tiers by Type', fontweight='bold', fontsize=13)
ax.set_xlabel('Equipment Type', fontsize=11)
ax.set_ylabel('Number of Units', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Priority Tier', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('uplift_type_tiers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_type_tiers.png')

## 6. Generate Executive Summary Markdown

In [ ]:
md_content = f"""# UpLift — Executive Summary
**Report Date:** {REPORT_DATE}  
**Decision Threshold:** {THRESHOLD}  
**Model:** XGBoost Supervised Classification  
**Produced by:** Nico Meyering, MPA — [github.com/meyeringn/uplift-transit](https://github.com/meyeringn/uplift-transit)

---

## What UpLift Found

UpLift scored **{len(report_df):,} equipment units** across {report_df['agency'].nunique()} transit agencies.

Of those, **{int(flagged):,} units** ({flagged/len(report_df):.1%}) are flagged for proactive service before a failure occurs.

| Priority Tier | Units | Recommended Action |
|---|---|---|
| 🔴 Critical | {int(critical)} | Dispatch service team within 72 hours |
| 🟠 High | {int(high)} | Schedule proactive service within 2 weeks |
| 🟡 Elevated | {int(elevated)} | Schedule inspection within 30 days |
| 🟢 Monitor | {len(report_df) - int(flagged)} | Observe at next routine maintenance cycle |

---

## Model Performance

| Metric | Value | What It Means |
|---|---|---|
| Recall | {recall:.1%} | Of real failures that occurred, UpLift caught {recall:.1%} of them in advance |
| Precision | {precision:.1%} | Of units flagged, {precision:.1%} actually experienced a failure |
| F2-Score | {f2:.3f} | Primary metric — weights recall twice as heavily as precision |
| AUC-ROC | {auc:.3f} | Overall model discrimination (1.0 = perfect, 0.5 = random) |

---

## The Core Judgment

A false positive — flagging a unit that turns out to be fine — costs an unnecessary service call (~$1,500).  
A false negative — missing a failure — means a Disabled rider is stranded, and the agency faces potential ADA enforcement.

These are not equal costs. UpLift is calibrated to minimize false negatives.

---

## Next Steps for Agency Partners

1. **Share your real equipment data** — UpLift can be retrained on agency-specific records in one session
2. **Select your operating threshold** — the full trade-off curve is in `outputs/uplift_threshold_sensitivity.png`
3. **Integrate with existing work order systems** — `uplift_agency_report.csv` is formatted for direct import

Contact: **nicmeyering@gmail.com** · [LinkedIn](https://www.linkedin.com/in/nicolasmeyering)
"""

with open('uplift_executive_summary.md', 'w') as f:
    f.write(md_content)

print('Saved: uplift_executive_summary.md')
print()
print(md_content)

## Summary — All Outputs From This Notebook

| File | Audience | Description |
|---|---|---|
| `uplift_agency_report.csv` | Maintenance schedulers | Full ranked equipment list with recommended actions |
| `uplift_agency_tiers.png` | Agency leadership | Risk tier breakdown by agency |
| `uplift_type_tiers.png` | Maintenance managers | Risk tier breakdown by equipment type |
| `uplift_executive_summary.md` | Any stakeholder | Plain-language summary ready for reports and proposals |

---

**UpLift is ready for real data.** Replace `uplift_equipment_data.csv` in the `/data` folder with your agency's records, rerun notebooks 01–06, and all outputs update automatically.

*Equitech Futures Civic Tech Institute, CTI 2026 · Philadelphia, PA*